In [210]:
from ket import *
from IPython.display import display

from ket import __version__
print(f"Ket v{__version__}")

Ket v0.9.3.5.post1


# Introdução
Computação Quântica - FSC7152

Primeiro, será apresentado a implementação do circuito SWAP Test isoladamente. Em seguida será mostrada a implementação do SWAP Test sob múltiplas iterações, a fim de gerar o resultado da probabilidade obtido após várias medidas. A seguir, uma série de exemplos são realizados.

## Seção 1: Implementando o SWAP Test

O objetivo é implementar a função SWAP Test, que será usada posteriormente para avaliar a similaridade entre duas mensagens de $n$ qubits.

O circuito do SWAP Test é definido como:


In [211]:
def swap_test(m1: Quant, m2: Quant, ancilla: Quant):
    with around(H, ancilla):
        for i in range(len(m1)):
            ctrl(ancilla, SWAP)(m1[i], m2[i])
    return measure(ancilla).value

## Seção 2: Implementando o Multiple SWAP Test

O SWAP Test retorna um resultado probabílistico. Os resultados de múltiplas medidas devem ser utilizados para se aproximar da probabilidade real de retorno da função SWAP Test. Para isso, introduzimos a próxima função, que recebe várias cópias de cada mensagem. A função SWAP Test é chamada então para cada par de cópias m1 e m2. Os resultados então são contados com fim de obter-se a probabilidade estimada.

In [212]:
def multiple_swap_test(m1_copies: list[Quant], m2_copies: list[Quant], ancilla: Quant, num_copies : int):
    m_zero = 0
    for i in range(num_copies):
        if (swap_test(m1_copies[i], m2_copies[i], ancilla)):
            m_zero += 1
            X(ancilla)
    return m_zero / num_copies

## Seção 3: Casos de Teste

Com as funções já implementadas, devemos agora verficar suas corretudes. Para isso, uma série de teste foram prepardos.

**SWAP Test:**

O código abaixo imprime o resultado da realização de um SWAP Test em dois estados completamente ortogonais. Verifica-se que ora obtém-se 0, ora obtém-se 1 como resposta.

In [213]:
n = 10
processo = Process()
ancilla = processo.alloc()
m1 = processo.alloc(n)
m2 = processo.alloc(n)

for i in range (n):
    X(m2[i])

result = swap_test(m1, m2, ancilla)

print(result)

1


O próximo exemplo avalia estados equivalentes. Nota-se que o resultado é sempre 0.

In [216]:
n = 10
processo = Process()
ancilla = processo.alloc()
m1 = processo.alloc(n)
m2 = processo.alloc(n)

result = swap_test(m1, m2, ancilla)

print(result)

0


**Multiple SWAP Test:**

Agora, iremos testar o código que realiza múltiplas chamadas ao SWAP test, com objetivo de juntar todos os resultados obtidos para montar a probabilidade de medida. A seguir está o código de teste para dois estados ortogonais. Para tal, é necessário fazer diversas cópias dos estados iniciais. Verificamos que quanto maior o número de cópias, mais o resultado se aproxima de 0.5, como esperado. Note que, quando a ancilla colapsa para $\ket{1}$, precisamos negá-la antes de proceder para a próxima medida.

In [218]:
n = 1
num_copies = 10

processo = Process()
ancilla = processo.alloc()

m1_copies = []
m2_copies = []

for i in range(num_copies):
    m1 = processo.alloc(n)
    m1_copies.append(m1)
    m2 = processo.alloc(n)
    m2_copies.append(m2)

for c in range(num_copies):
    for i in range(n):
        X(m2_copies[c][i])

result = multiple_swap_test(m1_copies, m2_copies, ancilla, num_copies)

print(result)

0.6
